In [1]:
import pandas as pd
import numpy as np

In [2]:
bureau= pd.read_csv("D:/AI/HomeCredit/Dataset/bureau.csv")
bb = pd.read_csv("D:/AI/HomeCredit/Dataset/bureau_balance.csv")

In [3]:
bb['STATUS'].unique()

array(['C', '0', 'X', '1', '2', '3', '5', '4'], dtype=object)

In [4]:
status_map = {'0':0,'1':1,'2':2,'3':3,'4':4,'5':5,'C':0,'X':0}
bb['STATUS_NUM'] = bb['STATUS'].map(status_map)

In [5]:
bb_agg = bb.groupby('SK_ID_BUREAU').agg(
    bb_months_min = ('MONTHS_BALANCE','min'),
    bb_months_max = ('MONTHS_BALANCE','max'),
    bb_months_count = ('MONTHS_BALANCE', 'count'),
    bb_max_status = ('STATUS_NUM', 'max'),
    bb_sum_status = ('STATUS_NUM', 'sum'),
    bb_overdue_months = ('STATUS_NUM', lambda x: (x>0).sum()),
    bb_severe_overdue_months = ('STATUS_NUM', lambda x: (x>=3).sum())).reset_index()

In [6]:
bb_agg

,SK_ID_BUREAU,bb_months_min,bb_months_max,bb_months_count,bb_max_status,bb_sum_status,bb_overdue_months,bb_severe_overdue_months
0,5001709,-96,0,97,0,0,0,0
1,5001710,-82,0,83,0,0,0,0
2,5001711,-3,0,4,0,0,0,0
3,5001712,-18,0,19,0,0,0,0
4,5001713,-21,0,22,0,0,0,0
...,...,...,...,...,...,...,...,...
817390,6842884,-47,0,48,0,0,0,0
817391,6842885,-23,0,24,5,60,12,12
817392,6842886,-32,0,33,0,0,0,0
817393,6842887,-36,0,37,0,0,0,0


In [8]:
bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how = 'left')

In [9]:
bureau.isna().sum()

SK_ID_CURR                       0
SK_ID_BUREAU                     0
CREDIT_ACTIVE                    0
CREDIT_CURRENCY                  0
DAYS_CREDIT                      0
CREDIT_DAY_OVERDUE               0
DAYS_CREDIT_ENDDATE          65119
DAYS_ENDDATE_FACT           389308
AMT_CREDIT_MAX_OVERDUE      690835
CNT_CREDIT_PROLONG               0
AMT_CREDIT_SUM                   7
AMT_CREDIT_SUM_DEBT         157335
AMT_CREDIT_SUM_LIMIT        365062
AMT_CREDIT_SUM_OVERDUE           0
CREDIT_TYPE                      0
DAYS_CREDIT_UPDATE               0
AMT_ANNUITY                 729443
bb_months_min               545543
bb_months_max               545543
bb_months_count             545543
bb_max_status               545543
bb_sum_status               545543
bb_overdue_months           545543
bb_severe_overdue_months    545543
dtype: int64

In [10]:
num_zero_fill = ['AMT_CREDIT_MAX_OVERDUE', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_OVERDUE', 'AMT_CREDIT_SUM_LIMIT', 'AMT_ANNUITY']
bureau[num_zero_fill] = bureau[num_zero_fill].fillna(0)

In [11]:
bureau['HAS_CREDIT_OVERDUE'] = (bureau['CREDIT_DAY_OVERDUE']>0).astype(int)
bureau['HAS_CREDIT_PROLONG'] =(bureau['CNT_CREDIT_PROLONG']>0).astype(int)
bureau['IS_ACTIVE_CREDIT'] = (bureau['CREDIT_ACTIVE']=='Active').astype(int)

In [12]:
bureau['CREDIT_AGE'] = -bureau['DAYS_CREDIT']
bureau['CREDIT_END_GAP'] = bureau['DAYS_CREDIT_ENDDATE'] - bureau['DAYS_CREDIT']

In [13]:
cols_to_agg = ['AMT_CREDIT_SUM',
    'AMT_CREDIT_SUM_DEBT',
    'AMT_CREDIT_SUM_OVERDUE',
    'AMT_CREDIT_SUM_LIMIT',
    'AMT_ANNUITY',
    'CREDIT_DAY_OVERDUE',
    'CREDIT_AGE',
    'CREDIT_END_GAP',
    'bb_months_count',
    'bb_overdue_months',
    'bb_severe_overdue_months',
    'bb_max_status']
bureau_agg = bureau.groupby('SK_ID_CURR')[cols_to_agg].agg(['mean','sum','max'])
bureau_agg.columns = ['_'.join(col).upper() for col in bureau_agg.columns]
bureau_agg.reset_index(inplace=True)
bureau_agg

,SK_ID_CURR,AMT_CREDIT_SUM_MEAN,AMT_CREDIT_SUM_SUM,AMT_CREDIT_SUM_MAX,AMT_CREDIT_SUM_DEBT_MEAN,AMT_CREDIT_SUM_DEBT_SUM,AMT_CREDIT_SUM_DEBT_MAX,AMT_CREDIT_SUM_OVERDUE_MEAN,AMT_CREDIT_SUM_OVERDUE_SUM,AMT_CREDIT_SUM_OVERDUE_MAX,...,BB_MONTHS_COUNT_MAX,BB_OVERDUE_MONTHS_MEAN,BB_OVERDUE_MONTHS_SUM,BB_OVERDUE_MONTHS_MAX,BB_SEVERE_OVERDUE_MONTHS_MEAN,BB_SEVERE_OVERDUE_MONTHS_SUM,BB_SEVERE_OVERDUE_MONTHS_MAX,BB_MAX_STATUS_MEAN,BB_MAX_STATUS_SUM,BB_MAX_STATUS_MAX
0,100001,207623.571429,1453365.000,378000.0,85240.928571,596686.50,373239.0,0.0,0.0,0.0,...,52.0,0.142857,1.0,1.0,0.0,0.0,0.0,0.142857,1.0,1.0
1,100002,57925.927500,347555.565,135000.0,0.000000,0.00,0.0,0.0,0.0,0.0,...,22.0,2.500000,15.0,6.0,0.0,0.0,0.0,0.666667,4.0,1.0
2,100003,254350.125000,1017400.500,810000.0,0.000000,0.00,0.0,0.0,0.0,0.0,...,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN
3,100004,94518.900000,189037.800,94537.8,0.000000,0.00,0.0,0.0,0.0,0.0,...,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN
4,100008,186655.500000,373311.000,267606.0,120028.500000,240057.00,240057.0,0.0,0.0,0.0,...,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218287,456243,308770.200000,1543851.000,1282500.0,249490.800000,1247454.00,1247454.0,0.0,0.0,0.0,...,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN
218288,456247,677874.857143,4745124.000,3118500.0,313341.428571,2193390.00,2193390.0,0.0,0.0,0.0,...,29.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
218289,456249,339719.040000,1358876.160,540000.0,0.000000,0.00,0.0,0.0,0.0,0.0,...,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN
218290,456253,990000.000000,3960000.000,2250000.0,448958.250000,1795833.00,1624797.0,0.0,0.0,0.0,...,31.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


In [14]:
bureau_cat = pd.get_dummies(bureau[['SK_ID_CURR','CREDIT_ACTIVE','CREDIT_TYPE']], columns = ['CREDIT_ACTIVE', 'CREDIT_TYPE'])
bureau_cat_agg = bureau_cat.groupby('SK_ID_CURR').mean().reset_index()
bureau_final = bureau_agg.merge(bureau_cat_agg, on='SK_ID_CURR', how='left')

In [15]:
bureau_final

,SK_ID_CURR,AMT_CREDIT_SUM_MEAN,AMT_CREDIT_SUM_SUM,AMT_CREDIT_SUM_MAX,AMT_CREDIT_SUM_DEBT_MEAN,AMT_CREDIT_SUM_DEBT_SUM,AMT_CREDIT_SUM_DEBT_MAX,AMT_CREDIT_SUM_OVERDUE_MEAN,AMT_CREDIT_SUM_OVERDUE_SUM,AMT_CREDIT_SUM_OVERDUE_MAX,...,CREDIT_TYPE_Interbank credit,CREDIT_TYPE_Loan for business development,CREDIT_TYPE_Loan for purchase of shares (margin lending),CREDIT_TYPE_Loan for the purchase of equipment,CREDIT_TYPE_Loan for working capital replenishment,CREDIT_TYPE_Microloan,CREDIT_TYPE_Mobile operator loan,CREDIT_TYPE_Mortgage,CREDIT_TYPE_Real estate loan,CREDIT_TYPE_Unknown type of loan
0,100001,207623.571429,1453365.000,378000.0,85240.928571,596686.50,373239.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
1,100002,57925.927500,347555.565,135000.0,0.000000,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
2,100003,254350.125000,1017400.500,810000.0,0.000000,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
3,100004,94518.900000,189037.800,94537.8,0.000000,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4,100008,186655.500000,373311.000,267606.0,120028.500000,240057.00,240057.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218287,456243,308770.200000,1543851.000,1282500.0,249490.800000,1247454.00,1247454.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
218288,456247,677874.857143,4745124.000,3118500.0,313341.428571,2193390.00,2193390.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.142857,0.0,0.0
218289,456249,339719.040000,1358876.160,540000.0,0.000000,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
218290,456253,990000.000000,3960000.000,2250000.0,448958.250000,1795833.00,1624797.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


In [16]:
bureau_final['DEBT_TO_CREDIT_RATIO'] = (
    bureau_final['AMT_CREDIT_SUM_DEBT_SUM'] /
    (bureau_final['AMT_CREDIT_SUM_SUM'] + 1e-6)
)
bureau_final['SEVERE_OVERDUE_RATIO'] = (
    bureau_final['BB_SEVERE_OVERDUE_MONTHS_SUM'] /
    (bureau_final['BB_MONTHS_COUNT_SUM'] + 1e-6)
)


In [17]:
bureau_final.to_csv("bureau_final.csv",index=False)